In [16]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-opus-4-1-20250805"

In [24]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    print(message)              # Entire response
    print(message.content)      # Content blocks
    print(message.stop_reason)


    # return message

    if not message.content:
        raise RuntimeError("Model returned no content")
    return message.content[0].text



In [25]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [26]:
dataset = generate_dataset()
dataset

Message(id='msg_011CczCve8M52gynGpngt9Fm', container=None, content=[TextBlock(citations=None, text='\n[\n    {\n        "task": "Write a Python function that takes an AWS S3 bucket name and object key as parameters, and returns a pre-signed URL that expires in 1 hour using boto3"\n    },\n    {\n        "task": "Create a JSON IAM policy document that allows read-only access to all objects in a specific S3 bucket named \'my-data-bucket\' but denies access to objects with the prefix \'sensitive/\'"\n    },\n    {\n        "task": "Write a regular expression that validates AWS ARN (Amazon Resource Name) format, matching patterns like \'arn:aws:s3:::bucket-name/key\' or \'arn:aws:iam::123456789012:user/username\'"\n    }\n]\n', type='text')], model='claude-opus-4-1-20250805', role='assistant', stop_details=None, stop_reason='stop_sequence', stop_sequence='```', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation

[{'task': 'Write a Python function that takes an AWS S3 bucket name and object key as parameters, and returns a pre-signed URL that expires in 1 hour using boto3'},
 {'task': "Create a JSON IAM policy document that allows read-only access to all objects in a specific S3 bucket named 'my-data-bucket' but denies access to objects with the prefix 'sensitive/'"},
 {'task': "Write a regular expression that validates AWS ARN (Amazon Resource Name) format, matching patterns like 'arn:aws:s3:::bucket-name/key' or 'arn:aws:iam::123456789012:user/username'"}]

In [28]:
with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [30]:
def run_prompt(test_case):
    """Mergers the prompt and test case input, then return result"""
    prompt = f"""Please solve the following task:
    {test_case["task"]}

    """

    messages = []
    add_user_message(messages, prompt)
    output= chat(messages)
    return output

In [31]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [36]:
def run_eval(dataset):
    """Load the dataset and calls run-test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results    

In [37]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)    

Message(id='msg_011CczErYAYZ4YKB5FqSnTHd', container=None, content=[TextBlock(citations=None, text='Here\'s a Python function that generates a pre-signed URL for an S3 object:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\nfrom datetime import datetime, timedelta\n\ndef generate_presigned_url(bucket_name, object_key, expiration=3600):\n    """\n    Generate a pre-signed URL for an S3 object.\n    \n    Parameters:\n    -----------\n    bucket_name : str\n        The name of the S3 bucket\n    object_key : str\n        The key (path) of the object in the S3 bucket\n    expiration : int, optional\n        Time in seconds for the pre-signed URL to remain valid (default: 3600 = 1 hour)\n    \n    Returns:\n    --------\n    str or None\n        The pre-signed URL as a string, or None if an error occurred\n    \n    Raises:\n    -------\n    ClientError\n        If there\'s an error generating the pre-signed URL\n    """\n    \n    # Create an S3 client\n    s3_cli

In [38]:
print(results)

[{'output': 'Here\'s a Python function that generates a pre-signed URL for an S3 object:\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError\nfrom datetime import datetime, timedelta\n\ndef generate_presigned_url(bucket_name, object_key, expiration=3600):\n    """\n    Generate a pre-signed URL for an S3 object.\n    \n    Parameters:\n    -----------\n    bucket_name : str\n        The name of the S3 bucket\n    object_key : str\n        The key (path) of the object in the S3 bucket\n    expiration : int, optional\n        Time in seconds for the pre-signed URL to remain valid (default: 3600 = 1 hour)\n    \n    Returns:\n    --------\n    str or None\n        The pre-signed URL as a string, or None if an error occurred\n    \n    Raises:\n    -------\n    ClientError\n        If there\'s an error generating the pre-signed URL\n    """\n    \n    # Create an S3 client\n    s3_client = boto3.client(\'s3\')\n    \n    try:\n        # Generate the pre-signed URL\n    